# 01 — Uploaded Dataset EDA

This notebook uses the uploaded `Diabetes Dataset(1).csv`. Inspect its real schema and categorical target rather than assuming it matches a different diabetes dataset.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
from src.preprocessing import quality_report, descriptive_statistics, candidate_clean
from src.visualization import plot_target, plot_distributions, plot_boxplots, plot_correlations, plot_missingness, plot_target_association
from src.data_loading import load_uploaded
df = load_uploaded()
target = "Outcome"
print("Shape:", df.shape, "| duplicate rows:", df.duplicated().sum())
display(df.head())
display(df.dtypes.rename("dtype").to_frame())

## Target distribution and quality

Class balance matters: a majority-class dataset can make a naïve classifier look deceptively useful. Missingness, duplicate rows, ranges and unique values should be reviewed before cleaning.

In [ ]:
display(df[target].value_counts(dropna=False).rename("count").to_frame())
display((df[target].value_counts(normalize=True, dropna=False)*100).round(2).rename("percent").to_frame())
display(quality_report(df))
display(descriptive_statistics(df))

## Plots

Histograms show distribution and skew; boxplots show spread and potential outliers; correlation is only a first-pass linear association for numeric columns.

In [ ]:
out = ROOT / "outputs" / "figures" / "uploaded"
plot_target(df, target, out)
plot_distributions(df, target, out)
plot_boxplots(df, target, out)
plot_missingness(df, out)
plot_correlations(df, target, out)
plot_target_association(df, target, out)
plt.show()

## Candidate preprocessing

This baseline imputes actual NaNs only (numeric median, categorical `Unknown`). It does not blindly replace zero values. For predictive modelling, learned imputers must be fitted on training data only.

In [ ]:
processed = candidate_clean(df)
print("Missing before:", int(df.isna().sum().sum()), "after:", int(processed.isna().sum().sum()))
display(quality_report(processed))
report_dir = ROOT / "outputs" / "reports" / "uploaded"
report_dir.mkdir(parents=True, exist_ok=True)
quality_report(df).to_csv(report_dir / "data_quality.csv", index=False)
descriptive_statistics(df).to_csv(report_dir / "descriptive_statistics.csv", index=False)